> ✍️ **My Work**

In [292]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


> ✍️ **My Work**

In [289]:
class Router(nn.Module): 
    def __init__(self, n = 10, d_model = 512): 
        super().__init__()
        self.W = nn.Linear(d_model , n, bias = True,  dtype = torch.float)
        nn.init.zeros_(self.W.weight)
        nn.init.zeros_(self.W.bias)
    def forward(self, x):
        # x is n experts to weigh 
        # x dim is ( B ,seq_len , d_model)
        # x @ W-> ( B, seq_len, d_model) x ( d_model,n) -> ( B, seq_len , n)
        return  self.W(x)


> ✍️ **My Work**

In [327]:
class MoE(nn.Module): 
    def __init__(self, n =10, d_model = 512, k = 5, w_load = 0.1, w_imp =0.1): 
        super().__init__()
        self.router = Router(n=n, d_model = d_model)
        # following the paper with starting this as a zeros array
        self.W_noise = nn.Linear(d_model, n)
        nn.init.zeros_(self.W_noise.weight)
        nn.init.zeros_(self.W_noise.bias)
        self.experts = nn.ModuleList(
            nn.Sequential(
                nn.Linear(d_model, d_model*4),
                nn.ReLU(),
                nn.Linear(d_model*4, d_model*4),
                nn.ReLU(), 
                nn.Linear(d_model*4 , d_model) 

            )
            for _ in range(n)

        )
    
        self.k =k
        self.n =n
        self.w_load = w_load
        self.w_imp = w_imp
        self.num_e = torch.zeros(size = (n, ),requires_grad = False, dtype = torch.long)
        
    def forward(self, x): 
        B, seq_len, _ = x.shape
        # x is (B, seq_len, d_model) 
        #Noisy Top-k Gating
        # logits -> (B, seq_len, n)
        router_logits= self.router(x)
        # apply the formula for H(x) logits + StandardNormal() * Softplus(x * W noise)
        sp_noise =F.softplus(self.W_noise(x)) 
        noise_term = torch.randn_like(router_logits)*sp_noise

        # print(router_logits.shape)
        H = router_logits +  noise_term
        # sort accross the last dim (B, seq_len, n) get top K
        # print(H.shape)
        keepTopK_plus_1 = torch.topk(input= H, k =self.k +1, dim = -1)
        keepTopK_indices = keepTopK_plus_1.indices[..., :-1]
        keepTopK_values = keepTopK_plus_1.values[..., :-1]
    

        # prepares for softmax all tensors that are not selected get 0 score
        inf_fill = torch.full_like(input = H, fill_value= float("-inf"))
        topKFilled =torch.scatter(input = inf_fill , dim =-1 , index=keepTopK_indices, src =keepTopK_values)
       
        #soft max to get G(x) = Softmax(KeepTopK(H(x), k))
        G = F.softmax(input = topKFilled , dim =-1)
        
        # print(G.shape)
        flatTopK = keepTopK_indices.flatten()

        #get counts per expert
        self.num_e = torch.scatter_reduce(input = self.num_e, dim =0, index =flatTopK, 
                    reduce= "sum", src = torch.ones_like(flatTopK, dtype = torch.long))
        
        # goal get all assigned inputs from x in a mini batch to loop over the expert 
        # we have G which is expert assigned weight per element 
        # we transform G to an array of size (n , B*seq_len) where True is the exact index where the expert is assigned 
        G_per_expert =G.flatten(0, 1).T 
        mask_for_assigned = G_per_expert>0
        assert mask_for_assigned.shape[0] == self.n, "Mismatch between experts and mask rows!"
        x_for_gather = x.flatten(0,1)
        # print(x_for_gather.shape)
        y = torch.zeros_like(x_for_gather)
        for i,e in enumerate(self.experts): 
            mask_for_expert = mask_for_assigned[i]
            tokens_for_expert = x_for_gather[mask_for_expert]
            if tokens_for_expert.numel() ==0: 
                continue
            # G_per_expert -> (n, B*seq_len) index per [i] and grab the ith mask to get the weights outputs 1d unsqueeze to broadcast 
            weight = G_per_expert[i][mask_for_assigned[i]].unsqueeze(-1)
            # same size as x for gather ->( B*seq_len, n ) need to map back to x dimension and scatter to the original location 
            output = weight* e(tokens_for_expert)
            y[mask_for_expert] += output


        y = y.reshape(B, seq_len , -1)
            
            
        # getting the importance of each expert as the batch-wise sum

        importance = G.sum(dim = (0,1))
      
        k_plus_1th = keepTopK_plus_1.values[..., -1].unsqueeze(-1)    
        kth=keepTopK_plus_1.values[..., - 2].unsqueeze(-1)
        is_in_top_k = H >= kth
        kth_excluding = torch.where(is_in_top_k, k_plus_1th, kth)
        P_preCDF = (router_logits - kth_excluding)/sp_noise
        normal_dist = torch.distributions.Normal(
                        loc=torch.tensor(0.0, device=x.device), 
                        scale=torch.tensor(1.0, device=x.device)
                    )
        # print(P_preCDF.shape)  
        P = normal_dist.cdf(P_preCDF)
        load = P.sum(dim = (0,1))
         
        #print(P.shape)

        # calculating losses 
        mean = load.mean()
        std = load.std()
        eps = 1e-8
        cv_load= std / (mean + eps)
        loss_load = self.w_load * cv_load**2
        mean = importance.mean()
        std = importance.std()
        eps = 1e-8
        cv_importance = std / (mean + eps)
        loss_importance= self.w_imp * cv_importance**2
        return y, loss_importance ,self.num_e, loss_load

> ✍️ **My Work**

In [328]:
d_model =10

m = MoE(d_model= d_model)
x = torch.randn((3, 12 , d_model))
x, I, E, L = m(x)
I ,  L

(tensor(0.0046, grad_fn=<MulBackward0>),
 tensor(4.2800e-05, grad_fn=<MulBackward0>))